# Scikit-learn Baseline Model

This notebook reproduces a traditional machine learning baseline for the binary diabetes risk prediction task.

The goal is to establish a reliable comparison point for the PyTorch models developed later in this project.

The baseline uses a scikit-learn preprocessing pipeline and a balanced Random Forest classifier, with evaluation focused on recall, ROC-AUC, threshold sensitivity, and class imbalance-aware performance.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

## Load Dataset

The dataset is loaded from the local `data/` directory.

The original target variable, `Diabetes_012`, contains three classes:

- 0: No diabetes
- 1: Prediabetes
- 2: Diabetes

Following the previous project, the target is converted into a binary clinical risk prediction task:

- 0: No diabetes
- 1: At risk, including prediabetes or diabetes

In [3]:
DATA_PATH = Path("../data/diabetes_012_health_indicators_BRFSS2015.csv")

def load_diabetes_data(data_path=DATA_PATH):
    if not DATA_PATH.is_file():
        raise FileNotFoundError(
            f"Dataset not found at {data_path}."
            "Please place the BRFSS 2015 diabetes csv file in the data/ directory."
        )
    return pd.read_csv(data_path)

df = load_diabetes_data()

print(df.shape)
print(df.head())

(253680, 22)
   Diabetes_012  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0           0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1           0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2           0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3           0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4           0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      5.0      18.0      15.0       1.0  0.0   9.0        

In [4]:
df["Diabetes_binary"] = (df["Diabetes_012"] > 0).astype(int)

target_distribution = df["Diabetes_binary"].value_counts(normalize=True).sort_index()

target_distribution

Diabetes_binary
0    0.842412
1    0.157588
Name: proportion, dtype: float64

## Train/Test Split

A stratified train/test split is used to preserve the proportion of non-risk and at-risk individuals in both sets.

This is important because the scikit-learn baseline and PyTorch models should be evaluated under the same class distribution. Keeping the test set consistent makes future model comparisons more fair.

Although the binary target and stratified split were introduced in the data understanding notebook, they are recreated here so that this baseline notebook can run independently and provide a reproducible training pipeline.

In [5]:
X = df.drop(columns=["Diabetes_012", "Diabetes_binary"])
y = df["Diabetes_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (202944, 21)
X_test: (50736, 21)


In [6]:
split_distribution = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index()
})

split_distribution

,train,test
Diabetes_binary,,
0,0.84241,0.84242
1,0.15759,0.15758
